# INF01090 - Ciência de Dados

# Lab Preparation Notebook — Visualization Techniques with Altair

This notebook is a **preparation guide** for the visualization laboratory in INF01090.

## Goals

By the end of this notebook, you should be able to:

- load and inspect a tabular dataset with pandas
- build basic charts with **Altair**
- choose suitable visual encodings for variables
- use aggregation, sorting, faceting, and interactivity
- critique and improve a chart
- use **vibe coding** productively, without losing understanding

This notebook uses a well-known public dataset so that you can focus on **visualization thinking**, not on searching for data.

## Dataset used in this notebook

We will use the **Cars** dataset available through `vega_datasets`.
It is small, clean, and excellent for practicing Altair because it contains:

- numerical attributes: horsepower, weight, acceleration, miles per gallon
- categorical attributes: origin
- temporal/grouping attributes: model year

This makes it useful for bar charts, histograms, scatter plots, faceting, filtering, and interactive analysis.

In [10]:
# If you are running this notebook for the first time, uncomment the next line:
# !pip install altair vega_datasets pandas

import pandas as pd
import altair as alt
from vega_datasets import data

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## 1. Load and inspect the data

Before making charts, always inspect the dataset.

Questions to ask:

- How many rows and columns are there?
- Which columns are numerical?
- Which columns are categorical?
- Are there missing values?
- Which columns look useful for comparison, trend, distribution, or relationship?

In [11]:
cars = data.cars()
cars.head()

,Name,Miles_per_Gallon,Cylinders,Displacement,Horsepower,Weight_in_lbs,Acceleration,Year,Origin
0,chevrolet chevelle malibu,18.0,8,307.0,130.0,3504,12.0,1970-01-01,USA
1,buick skylark 320,15.0,8,350.0,165.0,3693,11.5,1970-01-01,USA
2,plymouth satellite,18.0,8,318.0,150.0,3436,11.0,1970-01-01,USA
3,amc rebel sst,16.0,8,304.0,150.0,3433,12.0,1970-01-01,USA
4,ford torino,17.0,8,302.0,140.0,3449,10.5,1970-01-01,USA


In [8]:
cars.shape

(406, 9)

In [12]:
cars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 406 entries, 0 to 405
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Name              406 non-null    object        
 1   Miles_per_Gallon  398 non-null    float64       
 2   Cylinders         406 non-null    int64         
 3   Displacement      406 non-null    float64       
 4   Horsepower        400 non-null    float64       
 5   Weight_in_lbs     406 non-null    int64         
 6   Acceleration      406 non-null    float64       
 7   Year              406 non-null    datetime64[ns]
 8   Origin            406 non-null    object        
dtypes: datetime64[ns](1), float64(4), int64(2), object(2)
memory usage: 28.7+ KB


In [14]:
cars.describe()

,Miles_per_Gallon,Cylinders,Displacement,Horsepower,Weight_in_lbs,Acceleration,Year
count,398.000000,406.000000,406.000000,400.000000,406.000000,406.000000,406
mean,23.514573,5.475369,194.779557,105.082500,2979.413793,15.519704,1975-12-30 13:49:57.044334976
min,9.000000,3.000000,68.000000,46.000000,1613.000000,8.000000,1970-01-01 00:00:00
25%,17.500000,4.000000,105.000000,75.750000,2226.500000,13.700000,1973-01-01 00:00:00
50%,23.000000,4.000000,151.000000,95.000000,2822.500000,15.500000,1976-01-01 00:00:00
75%,29.000000,8.000000,302.000000,130.000000,3618.250000,17.175000,1979-01-01 00:00:00
max,46.600000,8.000000,455.000000,230.000000,5140.000000,24.800000,1982-01-01 00:00:00
std,7.815984,1.712160,104.922458,38.768779,847.004328,2.803359,NaN


In [15]:
cars.isna().sum()

Name                0
Miles_per_Gallon    8
Cylinders           0
Displacement        0
Horsepower          6
Weight_in_lbs       0
Acceleration        0
Year                0
Origin              0
dtype: int64

### Quick interpretation

Notice that the dataset mixes:

- **quantitative variables** like `Horsepower`, `Weight_in_lbs`, and `Miles_per_Gallon`
- **categorical variables** like `Origin`
- a **time/grouping variable**: `Year`

This is exactly the kind of structure that works well for Altair.

## 2. First chart: a bar chart

A bar chart is useful when we want to compare **counts** or **aggregated values** across categories.

Let's start with a simple question:

> How many cars come from each origin?

In [16]:
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Origin:N', title='Origin'),
    y=alt.Y('count():Q', title='Number of cars'),
    tooltip=['Origin:N', alt.Tooltip('count():Q', title='Count')]
).properties(
    title='Number of Cars by Origin',
    width=500,
    height=300
)

alt.Chart(...)

### Why this chart works

- `Origin` is categorical, so it goes naturally on one axis.
- We do not need a raw y-column; `count()` is computed automatically.
- The tooltip makes the chart easier to inspect.

### Practice prompt

Modify the chart so that bars are:
- sorted from highest count to lowest count
- colored by `Origin`

In [17]:
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Origin:N', sort='-y', title='Origin'),
    y=alt.Y('count():Q', title='Number of cars'),
    color=alt.Color('Origin:N', legend=None),
    tooltip=['Origin:N', alt.Tooltip('count():Q', title='Count')]
).properties(
    title='Number of Cars by Origin (Sorted)',
    width=500,
    height=300
)

alt.Chart(...)

## 3. Histograms: understanding distributions

Histograms help us understand the **distribution** of a numerical variable.

Question:

> How is `Miles_per_Gallon` distributed?

In [23]:
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Miles_per_Gallon:Q', bin=True, title='Miles per Gallon'),
    y=alt.Y('count():Q', title='Count'),
    tooltip=[alt.Tooltip('count():Q', title='Count')]
).properties(
    title='Distribution of Miles per Gallon',
    width=500,
    height=300
)

alt.Chart(...)

### Why this chart works

- `Miles_per_Gallon` is quantitative.
- Binning groups nearby values into intervals.
- Histograms are useful to see skewness, spread, and concentration.

### Practice prompt

Create a histogram for `Horsepower`.
Then compare:
- Is it symmetric?
- Is it skewed?
- Does it have a long tail?

In [24]:
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Horsepower:Q', bin=alt.Bin(maxbins=20), title='Horsepower'),
    y=alt.Y('count():Q', title='Count'),
    tooltip=[alt.Tooltip('count():Q', title='Count')]
).properties(
    title='Distribution of Horsepower',
    width=500,
    height=300
)

alt.Chart(...)

## 4. Scatter plots: relationships between variables

Scatter plots help us study the relationship between **two numerical variables**.

Question:

> What is the relationship between horsepower and miles per gallon?

In [25]:
alt.Chart(cars).mark_circle(size=60, opacity=0.65).encode(
    x=alt.X('Horsepower:Q', title='Horsepower'),
    y=alt.Y('Miles_per_Gallon:Q', title='Miles per Gallon'),
    color=alt.Color('Origin:N', title='Origin'),
    tooltip=['Name:N', 'Origin:N', 'Horsepower:Q', 'Miles_per_Gallon:Q']
).properties(
    title='Horsepower vs Miles per Gallon',
    width=550,
    height=350
)

alt.Chart(...)

### Interpretation

This chart often reveals a **negative relationship**:
cars with higher horsepower tend to have lower miles per gallon.

This is a good example of a chart that supports reasoning about **trade-offs**.

### Practice prompt

Try replacing:
- `Horsepower` with `Weight_in_lbs`
- `Miles_per_Gallon` with `Acceleration`

Then ask:
- Is the relationship strong?
- Is it linear?
- Are there clusters?

In [26]:
alt.Chart(cars).mark_circle(size=60, opacity=0.65).encode(
    x=alt.X('Weight_in_lbs:Q', title='Weight (lbs)'),
    y=alt.Y('Acceleration:Q', title='Acceleration'),
    color=alt.Color('Origin:N', title='Origin'),
    tooltip=['Name:N', 'Origin:N', 'Weight_in_lbs:Q', 'Acceleration:Q']
).properties(
    title='Weight vs Acceleration',
    width=550,
    height=350
)

alt.Chart(...)

## 5. Aggregation: from raw data to summaries

Sometimes we do not want to see all observations.
Instead, we want to compare **aggregated statistics**.

Question:

> What is the average miles per gallon by origin?

In [27]:
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Origin:N', title='Origin'),
    y=alt.Y('mean(Miles_per_Gallon):Q', title='Average MPG'),
    color=alt.Color('Origin:N', legend=None),
    tooltip=[
        'Origin:N',
        alt.Tooltip('mean(Miles_per_Gallon):Q', title='Average MPG', format='.2f')
    ]
).properties(
    title='Average Miles per Gallon by Origin',
    width=500,
    height=300
)

alt.Chart(...)

Aggregation is one of the most important ideas in data visualization.

The same dataset can be shown:
- at the **row level** (each car)
- at the **summary level** (mean, count, median, etc.)

A good analyst knows when detail is useful and when summary is clearer.

## 6. Boxplots: comparing distributions across groups

Averages are not enough.
Two groups can have the same average but very different spreads.

Question:

> How does horsepower vary across origins?

In [28]:
alt.Chart(cars).mark_boxplot().encode(
    x=alt.X('Origin:N', title='Origin'),
    y=alt.Y('Horsepower:Q', title='Horsepower'),
    color=alt.Color('Origin:N', legend=None),
    tooltip=['Origin:N']
).properties(
    title='Horsepower Distribution by Origin',
    width=500,
    height=320
)

alt.Chart(...)

### Why use a boxplot?

A boxplot summarizes:
- median
- quartiles
- spread
- possible outliers

It is useful when comparing **distributions across categories**.

## 7. Time or ordered trends

The dataset includes a year variable.
Let's convert it and examine a temporal trend.

Question:

> How did average miles per gallon change over time?

In [29]:
cars2 = cars.copy()
cars2['Year'] = pd.to_datetime(cars2['Year'])
cars2['Model_Year'] = cars2['Year'].dt.year

cars2[['Year', 'Model_Year']].head()

,Year,Model_Year
0,1970-01-01,1970
1,1970-01-01,1970
2,1970-01-01,1970
3,1970-01-01,1970
4,1970-01-01,1970


In [30]:
alt.Chart(cars2).mark_line(point=True).encode(
    x=alt.X('Model_Year:O', title='Model Year'),
    y=alt.Y('mean(Miles_per_Gallon):Q', title='Average MPG'),
    tooltip=[
        alt.Tooltip('Model_Year:O', title='Year'),
        alt.Tooltip('mean(Miles_per_Gallon):Q', title='Average MPG', format='.2f')
    ]
).properties(
    title='Average Miles per Gallon Over Time',
    width=600,
    height=320
)

alt.Chart(...)

### Interpretation

Now the visualization is about **trend**.
This kind of chart answers a different question than the previous charts.

A useful habit is to ask:

> Am I trying to compare categories, inspect distributions, or show change over time?

The answer strongly influences the chart type.

## 8. Faceting: small multiples

Faceting repeats a chart across groups.
This is often better than putting too many variables in a single plot.

Question:

> How does the horsepower vs miles-per-gallon relationship look within each origin?

In [31]:
alt.Chart(cars).mark_circle(size=50, opacity=0.6).encode(
    x=alt.X('Horsepower:Q', title='Horsepower'),
    y=alt.Y('Miles_per_Gallon:Q', title='Miles per Gallon'),
    tooltip=['Name:N', 'Horsepower:Q', 'Miles_per_Gallon:Q']
).properties(
    width=180,
    height=180
).facet(
    column=alt.Column('Origin:N', title='Origin')
).properties(
    title='Horsepower vs MPG, Faceted by Origin'
)

alt.FacetChart(...)

### Why faceting is useful

Faceting supports comparison while reducing clutter.
It is especially helpful when:

- color is not enough
- the groups overlap too much
- you want students to compare patterns carefully

## 9. Interactivity with selections

Altair is especially strong for interactive charts.

Below, a selection on the legend allows filtering by origin.

In [32]:
selection = alt.selection_point(fields=['Origin'], bind='legend')

alt.Chart(cars).mark_circle(size=70).encode(
    x=alt.X('Horsepower:Q', title='Horsepower'),
    y=alt.Y('Miles_per_Gallon:Q', title='Miles per Gallon'),
    color=alt.condition(selection, 'Origin:N', alt.value('lightgray')),
    tooltip=['Name:N', 'Origin:N', 'Horsepower:Q', 'Miles_per_Gallon:Q']
).add_params(
    selection
).properties(
    title='Interactive Scatter Plot by Origin',
    width=550,
    height=350
)

alt.Chart(...)

### Why this matters

Interactivity is not decoration.
It helps users:
- focus on subsets
- reduce overplotting
- ask follow-up questions

In a lab, this is one of the most important Altair features to practice.

## 10. Linked views

A powerful idea in visualization is to connect multiple charts.

Here, a brush selection in the scatter plot controls the bars.

In [33]:
brush = alt.selection_interval()

points = alt.Chart(cars).mark_circle(size=60, opacity=0.6).encode(
    x=alt.X('Horsepower:Q', title='Horsepower'),
    y=alt.Y('Miles_per_Gallon:Q', title='Miles per Gallon'),
    color=alt.condition(brush, 'Origin:N', alt.value('lightgray')),
    tooltip=['Name:N', 'Origin:N', 'Horsepower:Q', 'Miles_per_Gallon:Q']
).add_params(
    brush
).properties(
    width=360,
    height=300,
    title='Brush points'
)

bars = alt.Chart(cars).mark_bar().encode(
    x=alt.X('Origin:N', title='Origin'),
    y=alt.Y('count():Q', title='Selected cars'),
    color=alt.Color('Origin:N', legend=None)
).transform_filter(
    brush
).properties(
    width=220,
    height=300,
    title='Counts in selection'
)

points | bars

alt.HConcatChart(...)

### Interpretation

This example demonstrates **coordinated multiple views**.

A user selects points in one chart and sees a derived summary in another.
This is extremely powerful for exploratory analysis.

## 11. A note on chart design

Good visualizations are not just “correct code.”
They should also be readable and purposeful.

### Good habits

- choose a chart type that matches the question
- write informative titles
- label axes clearly
- avoid unnecessary color
- use tooltips when appropriate
- keep scales and encodings consistent
- prefer simple designs over decorative ones

### Bad habits

- too many encodings at once
- using pie charts when bar charts are clearer
- using colors that do not help interpretation
- making the reader guess what the chart is about

## 12. Using vibe coding productively

You are allowed to use AI assistance to help write code, but use it carefully.

### Good use of vibe coding

Ask for:
- an Altair scatter plot with tooltips
- a histogram with 20 bins
- a faceted plot by category
- an interactive selection example

### Bad use of vibe coding

Do **not** just copy and run code without checking:
- what each encoding means
- whether the chart answers the question
- whether the chart is readable
- whether the data types are correct

### Recommended prompt pattern

Use prompts like:

> "Generate an Altair chart from a pandas DataFrame named `df` with x = horsepower, y = mpg, color = origin, and tooltips. Add a title and use opacity for overplotting."

Then:
1. run it
2. inspect it
3. modify it yourself
4. explain why the chart works

## 13. Practice exercises (optional)

Try the following if you want to practice generating visualizations.

### Exercise 1
Create a bar chart showing the **average horsepower by origin**.

### Exercise 2
Create a histogram of `Weight_in_lbs`.

### Exercise 3
Create a scatter plot of `Weight_in_lbs` vs `Miles_per_Gallon`, colored by `Origin`.

### Exercise 4
Create a line chart showing average `Horsepower` by `Model_Year`.

### Exercise 5
Create a faceted histogram of `Miles_per_Gallon` by `Origin`.

### Exercise 6
Create one interactive chart using a legend selection or interval selection.

## 14. Suggested answers

Use these after you attempt the exercises on your own.

In [ ]:
# Exercise 1
alt.Chart(cars).mark_bar().encode(
    x='Origin:N',
    y='mean(Horsepower):Q',
    color=alt.Color('Origin:N', legend=None),
    tooltip=['Origin:N', alt.Tooltip('mean(Horsepower):Q', format='.2f')]
).properties(
    title='Average Horsepower by Origin',
    width=500,
    height=300
)

alt.Chart(...)

In [ ]:
# Exercise 2
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Weight_in_lbs:Q', bin=alt.Bin(maxbins=20), title='Weight (lbs)'),
    y=alt.Y('count():Q', title='Count')
).properties(
    title='Distribution of Weight',
    width=500,
    height=300
)

alt.Chart(...)

In [ ]:
# Exercise 3
alt.Chart(cars).mark_circle(size=60, opacity=0.65).encode(
    x=alt.X('Weight_in_lbs:Q', title='Weight (lbs)'),
    y=alt.Y('Miles_per_Gallon:Q', title='Miles per Gallon'),
    color='Origin:N',
    tooltip=['Name:N', 'Origin:N', 'Weight_in_lbs:Q', 'Miles_per_Gallon:Q']
).properties(
    title='Weight vs Miles per Gallon',
    width=550,
    height=350
)

alt.Chart(...)

In [ ]:
# Exercise 4
alt.Chart(cars2).mark_line(point=True).encode(
    x=alt.X('Model_Year:O', title='Model Year'),
    y=alt.Y('mean(Horsepower):Q', title='Average Horsepower')
).properties(
    title='Average Horsepower by Model Year',
    width=600,
    height=320
)

alt.Chart(...)

In [ ]:
# Exercise 5
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Miles_per_Gallon:Q', bin=alt.Bin(maxbins=15), title='Miles per Gallon'),
    y=alt.Y('count():Q', title='Count'),
    color=alt.Color('Origin:N', legend=None)
).properties(
    width=180,
    height=180
).facet(
    column='Origin:N'
).properties(
    title='MPG Distribution by Origin'
)

alt.FacetChart(...)

In [ ]:
# Exercise 6
sel = alt.selection_point(fields=['Origin'], bind='legend')

alt.Chart(cars).mark_circle(size=70).encode(
    x='Weight_in_lbs:Q',
    y='Horsepower:Q',
    color=alt.condition(sel, 'Origin:N', alt.value('lightgray')),
    tooltip=['Name:N', 'Origin:N', 'Weight_in_lbs:Q', 'Horsepower:Q']
).add_params(
    sel
).properties(
    title='Interactive Weight vs Horsepower',
    width=550,
    height=350
)

alt.Chart(...)